# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
%pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [3]:
#
#
# To find your Cloud ID and create an API key, 
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name='chatbot_rag_data', #give it a meaningful name
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [9]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name") or "Unknown Source"
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [10]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name='chatbot_rag_data',
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [14]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---

"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    normalized_docs = []
    for doc in docs:
        metadata = dict(doc.metadata or {})
        metadata.setdefault("name", "Unknown Source")
        doc.metadata = metadata
        normalized_docs.append(doc)

    doc_strings = [format_document(doc, document_prompt) for doc in normalized_docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

print("---- Answer ----")
ans = chain.invoke("what is the nasa sales team?")
print(ans)

---- Answer ----


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team operate within NASA?', '3. What are the responsibilities of the NASA sales team?']


The NASA sales team is a part of the Americas region in the sales organization. It is responsible for serving customers in North America and South America, with two Area Vice-Presidents, Laura Martinez and Gary Johnson. The team works closely with other departments to identify and pursue new business opportunities, maintain existing client relationships, and ensure customer satisfaction.


<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

In [15]:
my_questions = [
    # Questions about NASA sales team
    "What is the NASA sales team and what do they do?",
    "How does the NASA sales team work with commercial partners?",
    "What are the main responsibilities of the NASA sales organization?",
    "When was the NASA sales team established and why?",
    
    # Questions about commercial partnerships
    "What commercial space programs does NASA support?",
    "What partnerships has NASA formed with private companies?",
    "How does NASA's sales team help with space exploration missions?",
    
    # Questions about specific programs
    "What is the Commercial Crew Program?",
    "How does NASA work with SpaceX?",
    "What is NASA's role in commercial spaceflight?",
    
    # Questions about sales operations
    "Describe the structure of NASA's commercial partnerships.",
    "What is the process for companies to work with NASA?",
    "How does NASA generate revenue from commercial partnerships?",
]

print("📋CUSTOM QUESTIONS")
print(f"✅ Loaded {len(my_questions)} questions!")
print("\nCustom questions:")
for i, q in enumerate(my_questions, 1):
    print(f"   {i}. {q}")

📋CUSTOM QUESTIONS
✅ Loaded 13 questions!

Custom questions:
   1. What is the NASA sales team and what do they do?
   2. How does the NASA sales team work with commercial partners?
   3. What are the main responsibilities of the NASA sales organization?
   4. When was the NASA sales team established and why?
   5. What commercial space programs does NASA support?
   6. What partnerships has NASA formed with private companies?
   7. How does NASA's sales team help with space exploration missions?
   8. What is the Commercial Crew Program?
   9. How does NASA work with SpaceX?
   10. What is NASA's role in commercial spaceflight?
   11. Describe the structure of NASA's commercial partnerships.
   12. What is the process for companies to work with NASA?
   13. How does NASA generate revenue from commercial partnerships?


In [16]:
print("🔍 ASKING YOUR CUSTOM QUESTIONS")

all_results = []

for i, question in enumerate(my_questions, 1):
    print(f"\n{'─'*70}")
    print(f"❓ Question {i}: {question}")
    print(f"{'─'*70}")
    
    try:
        answer = chain.invoke(question)
        print(f"\n📝 Answer:\n{answer}")
        all_results.append({
            "question_number": i,
            "question": question,
            "answer": answer
        })
        
    except Exception as e:
        print(f"\n❌ Error: {e}")
        all_results.append({
            "question_number": i,
            "question": question,
            "answer": f"Error: {e}"
        })

print(f"\n✅ Completed {len(my_questions)} questions!")

🔍 ASKING YOUR CUSTOM QUESTIONS

──────────────────────────────────────────────────────────────────────
❓ Question 1: What is the NASA sales team and what do they do?
──────────────────────────────────────────────────────────────────────


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the role and responsibilities of the NASA sales team?', "2. How does the NASA sales team contribute to the organization's goals and objectives?", '3. What are the key functions and duties of the NASA sales team?']



📝 Answer:
The NASA sales team is a part of the Americas region in the sales organization. It is responsible for serving customers and achieving business objectives in North and South America. The team is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team consists of dedicated account managers, sales representatives, and support staff who work together to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction. They also collaborate with other departments, such as marketing, product development, and customer support, to consistently deliver high-quality products and services to clients.

──────────────────────────────────────────────────────────────────────
❓ Question 2: How does the NASA sales team work with commercial partners?
──────────────────────────────────────────────────────────────────────


INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the process for collaboration between the NASA sales team and commercial partners?', '2. Can you explain the dynamics of the relationship between the NASA sales team and commercial partners?', '3. How do commercial partners and the NASA sales team interact and cooperate with each other?']



📝 Answer:
The NASA sales team, which is responsible for the North America South America region, works closely with commercial partners to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction. This team is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America, and consists of dedicated account managers, sales representatives, and support staff. They collaborate closely with other departments, such as marketing, product development, and customer support, to ensure high-quality products and services are consistently delivered to clients.

──────────────────────────────────────────────────────────────────────
❓ Question 3: What are the main responsibilities of the NASA sales organization?
──────────────────────────────────────────────────────────────────────


INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the primary duties of the sales team at NASA?', '2. Can you list the key tasks of the NASA sales organization?', '3. What are the main obligations of the sales department at NASA?']


... (17 outputs left out)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. How can businesses collaborate with NASA?', '2. What are the steps for companies to partner with NASA?', '3. What is the procedure for corporations to engage with NASA?']



📝 Answer:

NASA, in this context, refers to the North America South America region within the sales organization. Companies can work with NASA by engaging with the Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America, who lead the sales team in this region. The sales team in this region is responsible for promoting and selling the company's products and services to potential clients in the United States, Canada, Mexico, Central America, and South America. Companies can also collaborate with the sales team in developing and refining sales materials, such as product presentations and technical documents, to ensure accurate and up-to-date information is presented to clients. Additionally, companies can work with the sales team on customer projects or product customizations, providing technical guidance and ensuring that the solutions meet the customer's requirements.

──────────────────────────────────────────────────────────────────────
❓ Question 13

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the methods used by NASA to generate revenue through commercial partnerships?', '2. Can you explain the ways in which NASA earns money from its commercial partnerships?', "3. How does NASA's revenue stream benefit from its partnerships with commercial entities?"]



📝 Answer:

NASA does not generate revenue from commercial partnerships. NASA is a government agency that focuses on space exploration and scientific research. They do not engage in commercial partnerships for revenue generation.

✅ Completed 13 questions!
